# 12E — Point-in-Time Quarterly Exposure Aggregation

This notebook converts validated document-level evidence into Q1–Q4 2024 firm signals. It uses deterministic rules frozen before returns or embedding-neighbour outcomes are opened. A 10-K is the baseline; later 10-Q evidence updates it, while silence does not erase it.

## 1. Load the frozen aggregation contract and point-in-time inputs

In [1]:
from pathlib import Path
import hashlib,json,numpy as np,pandas as pd
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name!='sp500_critical_minerals':
    candidate=ROOT/'geopolitical_exposure'/'sp500_critical_minerals'
    if candidate.exists(): ROOT=candidate
CONFIG=ROOT/'config'; DATA=ROOT/'data'/'processed'; SOURCE=ROOT/'resolved_extraction_v2_1'; OUT=ROOT/'quarterly_exposure_v1'; OUT.mkdir(exist_ok=True)
contract_path=CONFIG/'quarterly_exposure_aggregation_v1.json'; contract=json.loads(contract_path.read_text())
exposures=pd.read_parquet(SOURCE/'final_validated_document_exposures.parquet'); mapping=pd.read_parquet(DATA/'sp500_vintage_document_map.parquet'); universe=pd.read_parquet(DATA/'sp500_point_in_time_identifier_bridge.parquet')
mapping['filingDate']=pd.to_datetime(mapping.filingDate); mapping['cutoff']=pd.to_datetime(mapping.cutoff)
assert mapping.point_in_time_valid.all() if 'point_in_time_valid' in mapping else (mapping.filingDate<=mapping.cutoff).all()
print({'exposures':len(exposures),'documents':exposures.document_id.nunique(),'vintage_document_links':len(mapping),'returns_loaded':False,'embeddings_loaded':False})

{'exposures': 204, 'documents': 159, 'vintage_document_links': 3463, 'returns_loaded': False, 'embeddings_loaded': False}


## 2. Map each disclosure into every admissible quarterly information set

The vintage-document map already includes the applicable structural 10-K and available 10-Q updates for each firm and cutoff.

In [2]:
cols=['vintage','cutoff','permco','document_id','document_role','filingDate','reportDate']
panel=mapping[cols].merge(exposures,on='document_id',how='inner',validate='many_to_many')
assert (pd.to_datetime(panel.available_date)<=panel.cutoff).all()
mw=contract['materiality_weight']; dw=contract['directness_weight']; sw=contract['status_weight']
panel['eligible']=panel.materiality.isin(contract['eligible_materiality']) & panel.disclosure_status.isin(contract['eligible_status']) & panel.confidence.ge(contract['minimum_confidence']) & panel.exposure_strength.ge(contract['minimum_strength'])
panel['component_score']=panel.exposure_strength*panel.confidence*panel.materiality.map(mw).fillna(0)*panel.directness.map(dw).fillna(0)*panel.disclosure_status.map(sw).fillna(0)
panel.loc[~panel.eligible,'component_score']=0.0
panel=panel.sort_values(['vintage','permco','mechanism','direction','filingDate','component_score','document_id'],ascending=[True,True,True,True,False,False,True])
controlling=panel.drop_duplicates(['vintage','permco','mechanism','direction']).copy()
controlling.to_parquet(OUT/'quarterly_controlling_exposures.parquet',index=False)
display(controlling.groupby(['vintage','direction']).size().rename('records').reset_index())

,vintage,direction,records
0,2024q1,mixed,2
1,2024q1,negative,61
2,2024q1,positive,29
3,2024q1,unclear,3
4,2024q2,mixed,3
5,2024q2,negative,65
6,2024q2,positive,31
7,2024q2,unclear,5
8,2024q3,mixed,4
9,2024q3,negative,66


## 3. Construct separate positive and negative scores

The primary firm score is the maximum eligible component by direction. Mixed exposures contribute half their component score to both sides; unclear directions never create seeds.

In [3]:
controlling['positive_component']=np.where(controlling.direction.eq('positive'),controlling.component_score,np.where(controlling.direction.eq('mixed'),controlling.component_score*contract['mixed_direction_allocation'],0))
controlling['negative_component']=np.where(controlling.direction.eq('negative'),controlling.component_score,np.where(controlling.direction.eq('mixed'),controlling.component_score*contract['mixed_direction_allocation'],0))
scores=(controlling.groupby(['vintage','cutoff','permco'],as_index=False).agg(positive_score=('positive_component','max'),negative_score=('negative_component','max'),eligible_mechanisms=('eligible','sum'),disclosed_mechanisms=('mechanism','nunique'),latest_evidence_date=('filingDate','max'),evidence_documents=('document_id','nunique')))
base=universe[['vintage','cutoff','permco','ticker','comnam','cik']].drop_duplicates(['vintage','permco'])
quarterly=base.merge(scores,on=['vintage','cutoff','permco'],how='left',validate='one_to_one')
for c in ['positive_score','negative_score','eligible_mechanisms','disclosed_mechanisms','evidence_documents']: quarterly[c]=quarterly[c].fillna(0)
quarterly['net_score']=quarterly.positive_score-quarterly.negative_score
quarterly['positive_seed']=quarterly.positive_score.ge(0.30)
quarterly['negative_seed']=quarterly.negative_score.ge(0.30)
quarterly.to_parquet(OUT/'quarterly_firm_exposure_panel.parquet',index=False)
summary=quarterly.groupby('vintage',as_index=False).agg(universe_firms=('permco','nunique'),firms_with_disclosure=('disclosed_mechanisms',lambda x:int((x>0).sum())),positive_seeds=('positive_seed','sum'),negative_seeds=('negative_seed','sum'),mean_positive_score=('positive_score','mean'),mean_negative_score=('negative_score','mean'))
display(summary); summary.to_parquet(OUT/'quarterly_exposure_summary.parquet',index=False)

,vintage,universe_firms,firms_with_disclosure,positive_seeds,negative_seeds,mean_positive_score,mean_negative_score
0,2024q1,500,74,16,42,0.018637,0.040470
1,2024q2,500,75,15,41,0.017440,0.039831
2,2024q3,501,75,17,40,0.019492,0.039232
3,2024q4,500,76,17,41,0.020692,0.039668


## 4. Trace quarterly entry, persistence, and exit

These transitions describe the evidence signal, not portfolio performance. An apparent exit is flagged for later review because silence alone must not extinguish an earlier baseline.

In [4]:
qorder={'2024q1':1,'2024q2':2,'2024q3':3,'2024q4':4}; transitions=quarterly.sort_values(['permco','vintage']).copy(); transitions['q_order']=transitions.vintage.map(qorder)
transitions['prior_positive_seed']=transitions.groupby('permco').positive_seed.shift(); transitions['prior_positive_score']=transitions.groupby('permco').positive_score.shift()
transitions['positive_transition']=np.select([transitions.prior_positive_seed.isna(),~transitions.prior_positive_seed.fillna(False)&transitions.positive_seed,transitions.prior_positive_seed.fillna(False)&transitions.positive_seed,transitions.prior_positive_seed.fillna(False)&~transitions.positive_seed],['initial','entry','persistent','exit'],default='inactive')
transitions.to_parquet(OUT/'quarterly_signal_transitions.parquet',index=False)
display(transitions.groupby(['vintage','positive_transition']).size().rename('firms').reset_index())

/var/folders/0t/m56xtbls27l2yvm5frhjkf5h0000gn/T/ipykernel_23331/3233127698.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  transitions['positive_transition']=np.select([transitions.prior_positive_seed.isna(),~transitions.prior_positive_seed.fillna(False)&transitions.positive_seed,transitions.prior_positive_seed.fillna(False)&transitions.positive_seed,transitions.prior_positive_seed.fillna(False)&~transitions.positive_seed],['initial','entry','persistent','exit'],default='inactive')


,vintage,positive_transition,firms
0,2024q1,initial,500
1,2024q2,exit,1
2,2024q2,inactive,478
3,2024q2,initial,6
4,2024q2,persistent,15
5,2024q3,entry,2
6,2024q3,inactive,479
7,2024q3,initial,5
8,2024q3,persistent,15
9,2024q4,entry,1


## 5. Integrity gates and handoff

The next notebook may use these positive seeds for embedding propagation only after all point-in-time and coverage gates pass.

In [5]:
gate=pd.DataFrame([{'gate':'four_vintages','value':quarterly.vintage.nunique(),'threshold':4,'passed':quarterly.vintage.nunique()==4},{'gate':'unique_firm_vintage','value':int(quarterly.duplicated(['vintage','permco']).sum()),'threshold':0,'passed':not quarterly.duplicated(['vintage','permco']).any()},{'gate':'no_future_filings','value':int((pd.to_datetime(panel.available_date)>panel.cutoff).sum()),'threshold':0,'passed':not (pd.to_datetime(panel.available_date)>panel.cutoff).any()},{'gate':'positive_seeds_exist','value':int(quarterly.positive_seed.sum()),'threshold':1,'passed':quarterly.positive_seed.any()},{'gate':'returns_unopened','value':0,'threshold':0,'passed':True},{'gate':'embeddings_unopened','value':0,'threshold':0,'passed':True}])
display(gate); gate.to_parquet(OUT/'aggregation_decision_gates.parquet',index=False); assert gate.passed.all()
manifest={'stage':'quarterly_exposure_aggregation_v1','contract_sha256':hashlib.sha256(contract_path.read_bytes()).hexdigest(),'firm_quarters':len(quarterly),'positive_seed_firm_quarters':int(quarterly.positive_seed.sum()),'negative_seed_firm_quarters':int(quarterly.negative_seed.sum()),'returns_accessed':False,'embeddings_accessed':False}
(OUT/'aggregation_manifest.json').write_text(json.dumps(manifest,indent=2,sort_keys=True)+'\n'); print(json.dumps(manifest,indent=2))

,gate,value,threshold,passed
0,four_vintages,4,4,True
1,unique_firm_vintage,0,0,True
2,no_future_filings,0,0,True
3,positive_seeds_exist,65,1,True
4,returns_unopened,0,0,True
5,embeddings_unopened,0,0,True


{
  "stage": "quarterly_exposure_aggregation_v1",
  "contract_sha256": "f5e248c406ca5b715d9ed947b991f934f8ac1ddc6d510eae1a96c4db8de20313",
  "firm_quarters": 2001,
  "positive_seed_firm_quarters": 65,
  "negative_seed_firm_quarters": 164,
  "returns_accessed": false,
  "embeddings_accessed": false
}
